In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.ticker import MaxNLocator, NullLocator, FuncFormatter
from matplotlib import font_manager as fm
import sys, pathlib

sys.path.append(str(pathlib.Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")))
import Robyn_paper_2_defs
import Robyn_river_floods

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
catchments = gpd.read_file(base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg")[["catchment_uid","geometry"]]

In [ ]:
# mca_csv_path = base_path / "dphil_paper_2/results/bcr_mca_results//MCA_table_coding.xlsx"   # change if different
mca_csv_path = base_path / "dphil_paper_2/results/bcr_mca_results//catchment_mca_ranks_max.csv"   # change if different


In [ ]:
mca_table = pd.read_csv(mca_csv_path)
mca_table.head()

In [ ]:
mm = globals().get("mm", 1/25.4)

# --- prepare MCA table ---
mca_tbl = mca_table.copy().rename(columns={
    "Catchment": "catchment_uid",
    "MCA score": "mca_score",
    "MCA_score": "mca_score",
    "MCA": "mca_score",
})
mca_tbl["catchment_uid"] = pd.to_numeric(mca_tbl["catchment_uid"], errors="coerce").astype("Int64")
mca_tbl["mca_score"]     = pd.to_numeric(mca_tbl["mca_score"], errors="coerce")

gdf = catchments.merge(mca_tbl[["catchment_uid","mca_score"]], on="catchment_uid", how="left")

vals  = gdf["mca_score"].astype(float).to_numpy()
finite = vals[np.isfinite(vals)]
vmin = float(np.nanmin(finite)) if finite.size else 0.0
vmax = float(np.nanmax(finite)) if finite.size else 1.0
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
    vmin, vmax = 0.0, 1.0

cmap = mpl.colormaps["Greens_r"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

with mpl.rc_context(globals().get("NATURE_RC", {})):
    fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
    ax.set_axis_off()

    # --- CHANGED: black internal boundaries
    gdf.plot(
        ax=ax, column="mca_score", cmap=cmap, norm=norm,
        edgecolor="black", linewidth=0.45, zorder=1
    )
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

    # Coastline casing (unchanged)
    try:
        outline = getattr(jamaica_boundary, "union_all", getattr(jamaica_boundary, "unary_union"))()
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.2, zorder=98)
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
    except Exception:
        pass

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)
    cbar.set_label("MCA score", fontsize=6.5, labelpad=4)
    cbar.locator = MaxNLocator(nbins=6, integer=True)
    cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
    cbar.update_ticks()

    # ax.set_title("MCA score by catchment", fontsize=7, fontweight="bold", pad=6)

# Scale bar
    Robyn_paper_2_defs.draw_scale_bar(
    ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
    label_offset=0.02, km_offset=0.01
)

    # Labels
    NUM_FS = 5.0
    HALO_W = 0.75
    NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
    reps = gdf.geometry.representative_point()
    for (x, y, uid) in zip(reps.x, reps.y, gdf["catchment_uid"]):
        if pd.isna(uid):
            continue
        ax.text(x, y, str(int(uid)),
                fontproperties=NUM_FP, ha="center", va="center", color="black", zorder=20, snap=True,
                path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white", joinstyle="round", capstyle="round")])
        
    Robyn_paper_2_defs.draw_north_arrow(
    ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02
    )

    # Save
    fname = base_path / "dphil_paper_2/results/figures/fig_6_MCA_by_catchment_greens_lowdark_BLACKedges"
    fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", fname.with_suffix(".png"))

In [ ]:
mm = globals().get("mm", 1/25.4)

# --- prepare MCA table ---
mca_tbl = mca_table.copy().rename(columns={
    "Catchment": "catchment_uid",
    "MCA score": "mca_score",
    "MCA_score": "mca_score",
    "MCA": "mca_score",
})
mca_tbl["catchment_uid"] = pd.to_numeric(mca_tbl["catchment_uid"], errors="coerce").astype("Int64")
mca_tbl["mca_score"]     = pd.to_numeric(mca_tbl["mca_score"], errors="coerce")

gdf = catchments.merge(mca_tbl[["catchment_uid","mca_score"]], on="catchment_uid", how="left")

vals  = gdf["mca_score"].astype(float).to_numpy()
finite = vals[np.isfinite(vals)]
vmin = float(np.nanmin(finite)) if finite.size else 0.0
vmax = float(np.nanmax(finite)) if finite.size else 1.0
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
    vmin, vmax = 0.0, 1.0

cmap = mpl.colormaps["Greens_r"].with_extremes(bad=(0.92, 0.92, 0.92, 1.0))
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
    fig, ax = plt.subplots(figsize=(180*mm, 150*mm))
    ax.set_axis_off()

    # --- black internal boundaries
    gdf.plot(
        ax=ax, column="mca_score", cmap=cmap, norm=norm,
        edgecolor="black", linewidth=0.45, zorder=1
    )
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.45, zorder=15)

    # Coastline casing
    try:
        outline = getattr(jamaica_boundary, "union_all", getattr(jamaica_boundary, "unary_union"))()
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.2, zorder=98)
        gpd.GeoSeries([outline], crs=gdf.crs).plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)
    except Exception:
        pass

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)
    cbar.set_label("MCA score", fontsize=6.5, labelpad=4)
    cbar.locator = MaxNLocator(nbins=6, integer=True)
    cbar.formatter = FuncFormatter(lambda x, pos: f"{int(x)}")
    cbar.update_ticks()

    # Labels
    NUM_FS = 5.0
    HALO_W = 0.75
    NUM_FP = fm.FontProperties(family="Arial", size=NUM_FS)
    reps = gdf.geometry.representative_point()
    for (x, y, uid) in zip(reps.x, reps.y, gdf["catchment_uid"]):
        if pd.isna(uid):
            continue
        ax.text(x, y, str(int(uid)),
                fontproperties=NUM_FP, ha="center", va="center", color="black", zorder=20, snap=True,
                path_effects=[pe.withStroke(linewidth=HALO_W, foreground="white", joinstyle="round", capstyle="round")])

    # Scale bar + north arrow (true scale)
    if gdf.crs and gdf.crs.is_projected:
        pt = Robyn_paper_2_defs.add_scale_bar(
            ax, gdf, where="right-top",
            pad=0.07, length_km=20, max_frac=0.22,
            lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
        )
        if pt is not None:
            cx_data, cy_data = pt
            cx_ax, cy_ax = ax.transAxes.inverted().transform(
                ax.transData.transform((cx_data, cy_data))
            )
            Robyn_paper_2_defs.add_north_arrow_axes(
                ax, cx_ax, cy_ax,
                size_frac=0.080, gap_frac=0.050,
                shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
                fs=5, lw=0.5
            )

    # Save
    fname = base_path / "dphil_paper_2/results/figures/fig_6_MCA_by_catchment_greens_lowdark_BLACKedges"
    fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", fname.with_suffix(".png"))
